# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShardhaBatra/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*



### Research Question

Can historical search and engagement signals be used to identify content pages that are likely to experience future performance decline?

### Decision Supported

This work supports content teams in deciding which pages should be reviewed first for possible SEO or content improvement.

The system is intended as decision support, not as proof that updating a page will cause its performance to improve.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*



### Data Release

This project uses the FlyRank ML Internship warehouse release available on Hugging Face.

### Table Used

The main table used is `fact_content_daily_performance`, which contains daily search and engagement performance for content pages.

Each row represents the daily performance of one content page for one client on one date.

### Date Window

The analysis uses data from March 2026 through June 2026.

March 2026 is used for feature development and earlier periods are available for historical modeling. Later observations are used to construct and evaluate future outcomes using a time-aware design. June 2026 is treated as the final outcome/test period rather than being used to develop the label logic.

### Fields Excluded

Client and content IDs are used only for joining, grouping, and identifying rows, not as model features.

Future-window information, label-derived fields such as `trend_direction` and `trend_pct`, and product decision flags are excluded because they would reveal information about the outcome or introduce data leakage.

No client names, domains, URLs, private queries, credentials, or raw exports are used.

In [2]:
# Load the March 2026 warehouse data

from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset

login(userdata.get("HF_TOKEN"))

ds = load_dataset(
    "FlyRank/internship-warehouse",
    data_files="fact_content_daily_performance/month=2026-03/data_0.parquet",
    split="train",
    token=userdata.get("HF_TOKEN")
)

df = ds.to_pandas()

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Start date:", df["report_date"].min())
print("End date:", df["report_date"].max())

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Rows: 9841378
Columns: 30
Start date: 2026-03-01
End date: 2026-03-31


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [3]:
# Check the available warehouse months

months = [
    "2026-03",
    "2026-04",
    "2026-05",
    "2026-06"
]

print("Months available for capstone development:")
for month in months:
    print(month)

Months available for capstone development:
2026-03
2026-04
2026-05
2026-06


In [4]:
# Load April 2026 data

ds_april = load_dataset(
    "FlyRank/internship-warehouse",
    data_files="fact_content_daily_performance/month=2026-04/data_0.parquet",
    split="train",
    token=userdata.get("HF_TOKEN")
)

df_april = ds_april.to_pandas()

print("April rows:", len(df_april))
print("April start:", df_april["report_date"].min())
print("April end:", df_april["report_date"].max())

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  134MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

April rows: 10424730
April start: 2026-04-01
April end: 2026-04-30


In [5]:
# Check how many content pages appear in both March and April

march_content = set(df["content_hash_id"].unique())
april_content = set(df_april["content_hash_id"].unique())

common_content = march_content.intersection(april_content)

print("March unique content:", len(march_content))
print("April unique content:", len(april_content))
print("Content present in both months:", len(common_content))

March unique content: 331437
April unique content: 362172
Content present in both months: 331436


In [6]:
# Aggregate March performance for each content page

march_summary = (
    df.groupby("content_hash_id")
    .agg(
        march_impressions=("gsc_impressions", "sum"),
        march_clicks=("gsc_clicks", "sum")
    )
    .reset_index()
)

# Aggregate April performance for each content page

april_summary = (
    df_april.groupby("content_hash_id")
    .agg(
        april_impressions=("gsc_impressions", "sum"),
        april_clicks=("gsc_clicks", "sum")
    )
    .reset_index()
)

print("March content:", len(march_summary))
print("April content:", len(april_summary))

March content: 331437
April content: 362172


In [7]:
# Keep only content pages available in both months

comparison = march_summary.merge(
    april_summary,
    on="content_hash_id",
    how="inner"
)

print("Content available in both months:", len(comparison))

comparison.head()

Content available in both months: 331436


,content_hash_id,march_impressions,march_clicks,april_impressions,april_clicks
0,content_000005d4ced12088,86,0,81,0
1,content_00001e488b74b799,0,0,0,0
2,content_00007bd2985b77c3,47,0,48,0
3,content_00008950670cb6b5,0,0,0,0
4,content_0000a348850eb1fc,0,0,0,0


In [9]:
# Calculate percentage change in clicks from March to April
import pandas as pd
comparison["click_change_pct"] = (
    (comparison["april_clicks"] - comparison["march_clicks"])
    / comparison["march_clicks"].replace(0, pd.NA)
) * 100

print(comparison["click_change_pct"].describe())

count     68837.0
unique     3452.0
top        -100.0
freq      21205.0
Name: click_change_pct, dtype: float64


In [10]:
print("Pages with March clicks > 0:",
      (comparison["march_clicks"] > 0).sum())

print("Pages with March clicks = 0:",
      (comparison["march_clicks"] == 0).sum())

Pages with March clicks > 0: 68837
Pages with March clicks = 0: 262599


In [11]:
# Define future decline label

comparison["future_decline"] = (
    (comparison["march_clicks"] > 0) &
    (comparison["click_change_pct"] <= -30)
).astype(int)

print(comparison["future_decline"].value_counts())

future_decline
0    291676
1     39760
Name: count, dtype: int64


In [12]:
print(
    "Future decline rate:",
    round(comparison["future_decline"].mean() * 100, 2),
    "%"
)

Future decline rate: 12.0 %


Label Definition

A page is labeled as future_decline = 1 if its clicks decrease by at least 30% from March to April. This label uses future information only for evaluation and is never used as a model feature.

The decline rate is approximately 12%, meaning the dataset is moderately imbalanced.

In [13]:
# Build March-only features

features = march_summary.copy()

features = features.rename(columns={
    "march_impressions": "impressions",
    "march_clicks": "clicks"
})

# Add simple March features
features["ctr"] = (
    features["clicks"] /
    features["impressions"].replace(0, pd.NA)
) * 100

# Add the future label
features = features.merge(
    comparison[["content_hash_id", "future_decline"]],
    on="content_hash_id",
    how="inner"
)

print("Feature rows:", len(features))
print("Feature columns:")
print(features.columns.tolist())

features.head()

Feature rows: 331436
Feature columns:
['content_hash_id', 'impressions', 'clicks', 'ctr', 'future_decline']


,content_hash_id,impressions,clicks,ctr,future_decline
0,content_000005d4ced12088,86,0,0.0,0
1,content_00001e488b74b799,0,0,<NA>,0
2,content_00007bd2985b77c3,47,0,0.0,0
3,content_00008950670cb6b5,0,0,<NA>,0
4,content_0000a348850eb1fc,0,0,<NA>,0


In [15]:
# Build additional March-level features

march_features = (
    df
    .groupby("content_hash_id")
    .agg(
        avg_position=("gsc_avg_position", "mean"),
        ga4_sessions=("ga4_sessions", "sum"),
        scroll_events=("scroll_events", "sum")
    )
    .reset_index()
)

print(march_features.shape)
march_features.head()

(331437, 4)


,content_hash_id,avg_position,ga4_sessions,scroll_events
0,content_000005d4ced12088,72.854861,0.0,0.0
1,content_00001e488b74b799,NaN,0.0,0.0
2,content_00007bd2985b77c3,5.269565,0.0,0.0
3,content_00008950670cb6b5,NaN,2.0,1.0
4,content_0000a348850eb1fc,NaN,1.0,1.0


In [16]:
features = features.merge(
    march_features,
    on="content_hash_id",
    how="left"
)

print(features.shape)
print(features.columns.tolist())

(331436, 8)
['content_hash_id', 'impressions', 'clicks', 'ctr', 'future_decline', 'avg_position', 'ga4_sessions', 'scroll_events']


In [17]:
features.isnull().sum()

,0
content_hash_id,0
impressions,0
clicks,0
ctr,154699
future_decline,0
avg_position,154699
ga4_sessions,0
scroll_events,0


In [18]:
# Handle missing values

features["ctr"] = features["ctr"].fillna(0)
features["avg_position"] = features["avg_position"].fillna(0)

print(features.isnull().sum())

content_hash_id    0
impressions        0
clicks             0
ctr                0
future_decline     0
avg_position       0
ga4_sessions       0
scroll_events      0
dtype: int64


/tmp/ipykernel_4305/1683889082.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  features["ctr"] = features["ctr"].fillna(0)


In [19]:
# Separate features and target

X = features[
    [
        "impressions",
        "clicks",
        "ctr",
        "avg_position",
        "ga4_sessions",
        "scroll_events"
    ]
]

y = features["future_decline"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Decline rate:", round(y.mean() * 100, 2), "%")

X shape: (331436, 6)
y shape: (331436,)
Decline rate: 12.0 %


In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train decline rate:", round(y_train.mean() * 100, 2), "%")
print("Test decline rate:", round(y_test.mean() * 100, 2), "%")

Training rows: 265148
Test rows: 66288
Train decline rate: 12.0 %
Test decline rate: 12.0 %


In [21]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=8, n_jobs=-1,
                       random_state=42)

In [23]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

In [24]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("Precision:", round(precision_score(y_test, y_pred), 3))
print("Recall:", round(recall_score(y_test, y_pred), 3))
print("F1:", round(f1_score(y_test, y_pred), 3))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 3))

Precision: 0.582
Recall: 1.0
F1: 0.736
ROC-AUC: 0.966


In [25]:
# Build Week-4 baseline on the same test set

baseline_test = features.loc[X_test.index].copy()

baseline_test["baseline_score"] = 0

baseline_test.loc[
    baseline_test["impressions"] >= 100,
    "baseline_score"
] += 1

baseline_test.loc[
    baseline_test["clicks"] <= 5,
    "baseline_score"
] += 1

baseline_test.loc[
    baseline_test["avg_position"] >= 20,
    "baseline_score"
] += 1

baseline_test.loc[
    baseline_test["ga4_sessions"] <= 10,
    "baseline_score"
] += 1

baseline_test.loc[
    baseline_test["scroll_events"] <= 5,
    "baseline_score"
] += 1

print(baseline_test["baseline_score"].value_counts().sort_index())

baseline_score
1      735
2     1969
3    45413
4    14539
5     3632
Name: count, dtype: int64


In [26]:
baseline_pred = (
    baseline_test["baseline_score"] >= 4
).astype(int)

print("Baseline positive predictions:", baseline_pred.sum())
print("Actual declines:", y_test.sum())

Baseline positive predictions: 18171
Actual declines: 7952


In [27]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

print("Baseline Precision:",
      round(precision_score(y_test, baseline_pred), 3))

print("Baseline Recall:",
      round(recall_score(y_test, baseline_pred), 3))

print("Baseline F1:",
      round(f1_score(y_test, baseline_pred), 3))

Baseline Precision: 0.241
Baseline Recall: 0.552
Baseline F1: 0.336


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.